# Architecture Guide
## How to write good flexible code

In [ ]:

from dataclasses import dataclass
import numpy as np

## CLI flags become a config
cli flags are arguments you put when running main.
you can say main.py --controller pid:fast -- vision real:default

you basically are setting what type (of controller, of vision, of anything in the project) you want to use and with what preset

In [ ]:
# before running anything, we build the config from cli flags
config = {
    "controller": "pid:fast",
    "vision": "sim_cam:default",
    "dt": 0.004,
    ...
}
# it establishes the type of experiment we will do

## Classes
Every class needs two things, presets dict and params dataclass.

Dict has all presets to this class, maybe you want a faster controller, different poles, different estimator

Every dataclass joins what is in the dict into an organized object to be sent to the class itself.

In [ ]:
# class presets, many of them to choose, needs default
PID_PRESETS = {
    "default": {"kp": 1.0, "ki": 0.2, "kd": 0.05},
    "fast":    {"base": "default", "kp": 2.0},
}

# dataclass of init params for each class, you add a param here, you only change it in default preset
@dataclass
class PIDParams:
    kp: float
    ki: float
    kd: float

You basically want each class to receive a single thing, params

In [ ]:
class PIDController():
    def __init__(self, params: PIDParams):
        self.kp = params.kp
        self.ki = params.ki
        self.kd = params.kd

When you want to choose a dict, it needs to become into a dataclass to be able to be sent to the class, build params does that.

It takes the preset you have, builds a complete version if it has a "base", and then maps it to the params values. it returns the Params dataclass object

When build_from_registry does Params(**raw), Python's keyword argument unpacking matches _dict keys_ to _dataclass field names_ — case sensitive. So "kp" matches kp: float but "KP" would throw a TypeError.

In [ ]:
# create right preset if we have base
def resolve_preset(presets, name):
    p = presets[name]
    if "base" in p:
        base = resolve_preset(presets, p["base"])
        return {**base, **{k: v for k, v in p.items() if k != "base"}}
    return p


def build_params(Params, Presets, preset):
    raw = resolve_preset(Presets, preset)
    return Params(**raw)

### Spec
To init a class, you need:
- class type itself
- Params dataclass
- Presets to the class

These three values are enough to build any class.
For this reason we group them in a Spec object, and make a registry for all types of classes.

This makes any class easy to build by just seeing which type of class it is, what Params object it uses, and the Presets it has.

In [ ]:
@dataclass
class Spec:
    cls:        type
    Params:     type        # the dataclass
    Presets:    dict
    registries: dict | None = None # field_name -> registry, for nested objects
    sim_only:   bool | None = None # gotta revisit this

CONTROLLER_REGISTRY = {
    "pid": Spec(PIDController, PIDParams, PID_PRESETS),
    "lqr": Spec(LQRController, LQRParams, LQR_PRESETS),
}

# Building
With the registry alone, you can build anything, that's what build_from_registry is for. you give it the class registry and the type:preset you chose.

Nested classes are tricky because they require initializing classes inside, so they need the class registry of the classes they hold. That is what the registries attribute is for in Specs.

Make sure to make params know that you have objects or multiple objects (dict[str, class]) as attributes

You want to ask yourself

"class": "type:preset"

In [ ]:
SYSTEM_PRESETS = {
    
    "dynamic_sim": {
        "controllers": {"follower": "follower:default", "smooth": "smooth:default"},
        "estimators":  {"lpf": "lpf:default", "kalman": "kalman:default"},
        "sensor":     "sim_cam:default",
        "actuator":    "mock_servo:default",
        "supervisor":  "dynamic:default",   # <-- selects the state machine
    },
    "simple_sim": {
        "controllers": {"smooth": "smooth:default"},
        "estimators":  {"kalman": "kalman:default"},
        "actuator":    "mock_servo:default",
        "sensor":     "sim_cam:default",
        "supervisor":  "passthrough:default",   # <-- selects the state machine
    },
    "real": {"base": "dynamic_sim", "sensor": "dvs_cam:default", "actuator": "real_servo:default"},
}

@dataclass
class SystemParams:
    controllers: dict[str, Controller]  # {"follower": ..., "smooth": ...}
    estimators:  dict[str, Estimator]   # {"lpf": ..., "kalman": ...}
    sensor:      Sensor
    actuator:    Actuator
    supervisor:  Supervisor

SYSTEM_REGISTRY = {
    "default": Spec(
        cls      = System,
        Params   = SystemParams,
        Presets  = SYSTEM_PRESETS,
        registries = {
            "controllers": CONTROLLER_REGISTRY,  # dict of objects
            "estimators":  ESTIMATOR_REGISTRY,   # dict of objects
            "sensor":      SENSOR_REGISTRY,       # single object
            "actuator":    ACTUATOR_REGISTRY,
            "supervisor":  SUPERVISOR_REGISTRY,
        }
    )
}

# handles sub registries and injects params
def build_from_registry(registry, spec_string, **inject):
    type_, preset = spec_string.split(":")
    spec = registry[type_]
    raw = resolve_preset(spec.Presets, preset)

    resolved = {}
    for k, v in raw.items():
        sub_registry = (spec.registries or {}).get(k)
        if isinstance(v, str) and ":" in v:
            resolved[k] = build_from_registry(sub_registry, v)
        elif isinstance(v, dict) and sub_registry:
            resolved[k] = {name: build_from_registry(sub_registry, s) for name, s in v.items()}
        else:
            resolved[k] = v

    resolved.update(inject)   # injected params merge in after preset resolution
    return spec.cls(spec.Params(**resolved))

# to build multiple classes, create a system builder that builds in chunks
system = build_from_registry(SYSTEM_REGISTRY, config["system"])

Here is an example of the System class

In [ ]:
class System:
    def __init__(self, params: SystemParams):
        self.controllers = params.controllers   # dict[str, Controller]
        self.estimators  = params.estimators    # dict[str, Estimator]
        self.sensor      = params.sensor
        self.actuator    = params.actuator
        self.supervisor  = params.supervisor

        self.active_controller = self.controllers["follower"]
        self.active_estimator  = self.estimators["lpf"]
        self.state = None
        self.u     = None

    def step(self, dt):
        # 1. estimate and control with current actives
        x_est, innovation = self.active_estimator.estimate(self.sensor.read()) # estimator calculates innovation too
        u                 = self.active_controller.compute(x_est)
        
        self.actuator.apply(u)

        # 2. supervisor decides what should be active next step
        ctrl_key, est_key = self.supervisor.update(x_est, innovation, dt)

        # 3. system owns the swap — including warm-start on estimator switch
        new_estimator = self.estimators[est_key]
        if new_estimator is not self.active_estimator:
            new_estimator.initialize_from(self.active_estimator)

        self.active_controller = self.controllers[ctrl_key]
        self.active_estimator  = new_estimator
        self.state = x_est
        self.u     = u

So system is the physical things, but the experiment needs more abstract classes to help do it.

In [ ]:

EXPERIMENT_PRESETS = {
    "sim": {
        "system":          "default:sim",
        "logger":          "csv:default",
        "stop_condition":  "timeout:default",
        "visualizer":      {"realtime": "realtime:default", "animation": "animation:default"},
        "progress":        "progress_bar:default",
    },
    "real": {"base": "sim", "visualizer": "live:default"},
    "headless": {"base": "sim", "visualizer": "none:default"},
}

@dataclass
class ExperimentParams:
    system:         System                  # runs step
    logger:         Logger                  # logs data
    stop_condition: StopCondition           # determines when to stop simulation
    visualizer:     dict[str, Visualizer]   # does realtime render or post animation
    progress:       Progress                # shows progress bar of simulation, or simulations
    pacing:         Pacing                  # determines realtime or offline pacing?
    scheduler:      Scheduler               # when to actuate (if dt and actuator dt doesnt match) or when to render
    
    

EXPERIMENT_REGISTRY = {
    "default": Spec(
        cls      = Experiment,
        Params   = ExperimentParams,
        Presets  = EXPERIMENT_PRESETS,
        registries = {
            "system":          SYSTEM_REGISTRY,
            "logger":          LOGGER_REGISTRY,
            "stop_condition":  STOP_CONDITION_REGISTRY,
            "visualizer":      VISUALIZER_REGISTRY,
            "progress":        PROGRESS_REGISTRY,
            "pacing":          PACING_REGISTRY,
            "scheduler":       SCHEDULER_REGISTRY,
        }
    )
}

# main.py
experiment = build_from_registry(EXPERIMENT_REGISTRY, config["experiment"])
metrics = Metrics()
results = experiment.run()
summary = metrics.evaluate(results)
metrics.print_summary(summary)

Experiment wires up system runs it

In [ ]:

class Experiment:
    def run_trial(self):
        self.system.reset()
        while not self.stop_condition.should_stop(self.system.state):
            self.system.step()
            self.logger.log(self.system.state)
            self.visualizer.update(self.system.state)  # real-time hook (no-op in sim)
        self.visualizer.render(self.logger.get_data())  # post-run hook (no-op in real)
        
        return self.logger.get_result()

    def run_experiment(self):
        results = []
        
        for _ in range(self.n_trials):
            result = self.run_trial()
            results.append(result)
        
        return results

# Main 
Main just sets all the cli presets and overrides an builds from registry and runs it.

In [ ]:
def main():
    # 1. Parse CLI
    args = parse_args()

    # 2. Resolve config (preset + overrides)
    config = CONFIG_PRESETS[args.config].copy()

    if args.experiment:
        config["experiment"] = args.experiment

    # 3. Build and run
    experiment = build_from_registry(EXPERIMENT_REGISTRY, config["experiment"])
    results = experiment.run()

    # 4. Print / save
    print_summary(results)

main.py --config sim_lqr   # instead of --config sim --controller lqr:default

## Config Presets

Assembling a config from CLI flags can be tedius, so we can build some presets that assemble one for us

If we want to change one thing, it's as easy as calling the preset and overriding what we want

```
main.py --config sim --controller lqr:default
```


In [ ]:
CONFIG_PRESETS = {
    "sim": {
        "controller": "pid:default",
        "vision": "sim_cam:default",
        "dt": 0.002,
    }
}

## Incompatibility

The only thing that breaks stuff, is when we try to do an offline experiment with online classes. Montecarlo with real_dvs_cams? No can do. That's what the Spec sim_only attribute is used for. Most leave it as None, but actuator and vision use it to determine if an experiment can be run offline and therefore can be used in montecarlo simulations. You check it like so

In [ ]:
def get_type(spec_str):   return spec_str.split(":")[0]
def get_preset(spec_str): return spec_str.split(":")[1]

def is_fully_sim(config):
    # actuator: one level, sim_only lives directly on the spec
    actuator_type = get_type(config["actuator"])
    actuator_sim  = ACTUATOR_REGISTRY[actuator_type].sim_only

    # vision: two levels, sim_only lives on the interface spec inside
    vision_preset = get_preset(config["vision"])               # "sim_dvs_hough"
    vision_raw    = resolve_preset(VISION_PRESETS, vision_preset)  # the dict
    interface_type = get_type(vision_raw["interface"])          # "sim_dvs"
    vision_sim    = VISION_INTERFACE_REGISTRY[interface_type].sim_only

    return actuator_sim and vision_sim

## Changing config

You'll be tempted to run a config preset and change some params. Go ahead, but this is a dev tool. Users in the UI are not allowed to override anything other than controller and estimator because those two are compatible with all configs.

An example of being careful is using real_vision preset. this makes mock servo, connect to dvs_cams, and runs in realtime with a visualizer too. If you want to use sim_dvs, you can, and it will work essentially like a realtime_sim_dvs preset, but you won't have to remember or create that config preset. If you had used sim preset, you would have to change the actuator, the realtime scheduler, the visualization in realtime, turn off animation after experiment. Doable but more work. If this happens often, then creating a new config preset might be a good idea. Like real_vision and realtime_sim. 

# State Machine

We have gotten to the point of realism where I want to include a state machine, the path goes as follows

In [ ]:
@dataclass
class SupervisorParams:
    stable_threshold: float = 0.035  # ~2 deg in rad
    stable_hold_s:    float = 2.0
    consistent_hold_s:float = 1.0
    loss_threshold:   float = 0.3

class DynamicSupervisor:
    def __init__(self, params: SupervisorParams):
        self.params  = params
        self.state   = "ACQUISITION"
        self._t_state  = 0.0   # time in current state
        self._t_stable = 0.0   # continuous stable streak
        self._t_lost   = 0.0   # continuous lost streak

    def update(self, x_est, innovation, dt) -> tuple[str, str]:
        self._t_state += dt
        self._step(x_est, innovation, dt)
        return self._active()

    def _step(self, x_est, innovation, dt):
        # update streaks first, independent of state
        if self._is_stable(x_est):  self._t_stable += dt
        else:                        self._t_stable  = 0.0

        if self._is_lost(innovation): self._t_lost += dt
        else:                          self._t_lost  = 0.0

        s = self.state
        if s == "ACQUISITION":
            if self._t_stable >= self.params.stable_hold_s:
                self._transition("STABILIZATION_READY")

        elif s == "STABILIZATION_READY":
            if self._t_lost >= self.params.loss_hold_s:
                self._transition("ACQUISITION")
            elif self._t_state >= self.params.consistent_hold_s:
                self._transition("STABILIZING")

        elif s == "STABILIZING":
            if self._t_lost >= self.params.loss_hold_s:
                self._transition("ACQUISITION")
            elif self._t_state >= self.params.stable_hold_s:
                self._transition("BALANCED")

        elif s == "BALANCED":
            if self._t_lost >= self.params.loss_hold_s:
                self._transition("ACQUISITION")

    def _transition(self, new_state):
        self.state    = new_state
        self._t_state = 0.0
        # streaks carry over — losing for 2s before transition still counts

    def _active(self) -> tuple[str, str]:
        return {
            "ACQUISITION":         ("follower", "lpf"),
            "STABILIZATION_READY": ("follower", "lpf"),
            "STABILIZING":         ("smooth",   "kalman"),
            "BALANCED":            ("full",     "kalman"),
        }[self.state]

    def _is_stable(self, x_est):   return norm(x_est[:2]) < self.params.stable_threshold
    def _is_lost(self, innovation): return innovation is not None and norm(innovation) > self.params.loss_threshold

PRESET AND PARAMS  

In [ ]:
TIMING_PRESETS = {"default": {"total_time": 5.0, "dt": 4e-3}}

@dataclass
class TimingParams:
    total_time: float
    dt:         float

PLANT_PRESETS = {
    "default": {
        "g"             : 9.81,
        "l"             : 0.15,
        "tau"           : 0.03,
        "zeta"          : 0.8,
        "max_acc"       : 9.81 * 3,
        "num_states"    : 8,
        "x_ref"         : 0.0,
        "y_ref"         : 0.0,
        "safe_radius"   : 68e-3
    }
}

@dataclass
class PlantParams:
    g: float
    l: float
    tau: float
    zeta: float
    max_acc: float | None = None
    num_states: int
    x_ref: float
    y_ref: float
    safe_radius: float | None = None

POLE_PRESETS = {
    "default": {
        "plant": "default:default",          # controller owns its plant reference
        "poles": [-14, -16, -18, -20] * 2,
    }
}

@dataclass
class PoleParams:
    plant:  PlantParams   # physical constants live here, no copying
    poles:  list[float]
    
    
LQR_PRESETS = {
    "default": {
        "plant": "default:default",
        "Q_single_axis": np.diag([0.01, 0.01, 100, 10]),  # x, x_dot, alpha, alpha_dot
        "R":             np.eye(2) * 1e6,
    }
}

@dataclass
class LQRParams:
    plant:          PlantParams   # physical constants live here, no copying
    Q_single_axis:  np.ndarray
    R:              np.ndarray 



SMOOTH_POLE_PRESETS = {
    "default": {
        "plant":         "default:default",
        "timing":        "default:default",
        "s_poles":       [-14, -16, -18, -20] * 2,
        "slew_poles":    0.95 # 0 same as without it, 1 u frozen
    }
}

@dataclass
class SmoothPoleParams:
    plant:          PlantParams   # physical constants live here, no copying
    timing:         TimingParams
    s_poles:        list[float]
    slew_poles:     float 
    


SMOOTH_LQR_PRESETS = {
    "default": {
        "plant":            "default:default",
        "Q_single_axis":    np.diag([0.01, 0.01, 100, 10]),  # x, x_dot, alpha, alpha_dot
        "R":                np.eye(2) * 1e6,
        "q_u":              1e-6,
        "r_delta":          1e4,
    }
}

@dataclass
class SmoothLQRParams:
    plant:          PlantParams   # physical constants live here, no copying
    Q_single_axis:  np.ndarray
    q_u:            float
    r_delta:        float
    
CIRCLE_PRESETS = {
    "default": {
        "plant":         "default:default",
        "timing":        "default:default",
        "period_s":       18,
    }
}

@dataclass
class CircleParams:
    plant:          PlantParams   # physical constants live here, no copying
    timing:         TimingParams
    period_s:       float



Here are all registries, so it's clear

In [ ]:

PLANT_REGISTRY = {
    "sim": Spec(BalancerPlant, PlantParams, PLANT_PRESETS),
    "null": Spec(NullPlant, NullParams, NULL_PRESETS),
}

CONTROLLER_REGISTRY = {
    "pole": Spec(PoleController, PoleParams, POLE_PRESETS, registries={"plant": PLANT_REGISTRY}),
    "lqr": Spec(LQRController, LQRParams, LQR_PRESETS, registries={"plant": PLANT_REGISTRY}),
    "smooth_pole": Spec(SmoothPoleController, SmoothPoleParams, SMOOTH_POLE_PRESETS, registries={"plant": PLANT_REGISTRY}),
    "circle": Spec(CircleController, CircleParams, CIRCLE_PRESETS, registries={"plant": PLANT_REGISTRY}),
    "null": Spec(NullController, NullParams, NULL_PRESETS),
}

ESTIMATOR_REGISTRY = {
    "fde": Spec(FiniteDifferenceEstimator, NullParams, NULL_PRESETS),
    "lpf": Spec(LowPassFiniteDifferenceEstimator, LPFParams, LPF_PRESETS),
    "kalman": Spec(KalmanEstimator, KalmanParams, KALMAN_PRESETS),
    "full_kalman": Spec(FullStateKalmanFilter, FullKalmanParams, FULL_KALMAN_PRESETS),
}

LINE_ALGO_REGISTRY = {
    "hough": Spec(PaperHoughLineAlgorithm, HoughLineParams, HOUGH_PRESETS),
    "sam": Spec(SamLineAlgorithm, SamLineParams, SAM_PRESETS),
}

REG_MODEL_REGISTRY = {
    "none": Spec(NullRegression, NullParams, NULL_PRESETS),
    "simple": Spec(SimpleDVSRegressionModel, SimpleRegressionParams, SIMPLE_REG_PRESETS),
}

VISION_INTERFACE_REGISTRY = {
    "sim_analytic": Spec(SimVisionModel, SimAnalyticParams, SIM_ANALYTIC_PRESETS, sim_only=True),
    "sim_dvs": Spec(SimEventCameraInterface, SimDVSParams, SIM_DVS_PRESETS, sim_only=True),
    "real_dvs": Spec(RealEventCameraInterface, RealDVSParams, REAL_DVS_PRESETS, sim_only=False),
}

VISION_REGISTRY = {
    "default": Spec(
        cls       = Vision,
        Params    = VisionParams,
        Presets   = VISION_PRESETS,
        registries={
            "interface": VISION_INTERFACE_REGISTRY,
            "algo":      LINE_ALGO_REGISTRY,
            "reg_model": REG_MODEL_REGISTRY,
        },
    )
}

ACTUATOR_REGISTRY = {
    "servo": Spec(ServoController, ServoParams, SERVO_PRESETS, sim_only=False),
    "mock": Spec(MockServoController, NullParams, NULL_PRESETS, sim_only=True),
}

SUPERVISOR_REGISTRY = {
    "dynamic": Spec(DynamicSupervisor, DynamicSupervisorParams, DYNAMIC_SUPERVISOR_PRESETS), # multi states
    "static": Spec(StaticSupervisor, StaticSupervisorParams, STATIC_SUPERVISOR_PRESETS),  # single state
}

SYSTEM_REGISTRY = {
    "default": Spec(
        cls      = System,
        Params   = SystemParams,
        Presets  = SYSTEM_PRESETS,
        registries = {
            "plant":       PLANT_REGISTRY,
            "controllers": CONTROLLER_REGISTRY,  # dict of objects
            "estimators":  ESTIMATOR_REGISTRY,   # dict of objects
            "vision":      VISION_REGISTRY,       # composite; interface string + nested algo / reg_model
            "actuator":    ACTUATOR_REGISTRY,
            "supervisor":  SUPERVISOR_REGISTRY,
        }
    )
}

LOGGER_REGISTRY = {
    "default": Spec(Logger, NullParams, NULL_PRESETS),
}

STOP_CONDITION_REGISTRY = {
    "fall": Spec(FallCondition, FallConditionParams, FALL_CONDITION_PRESETS),
    "stabilized": Spec(StabilizedCondition, StabilizedParams, STABILIZED_CONDITION_PRESETS),
    "max_steps": Spec(MaxStepsCondition, MaxStepsConditionParams, MAX_STEPS_CONDITION_PRESETS),
    "any": Spec(AnyStopCondition, AnyStopConditionParams, ANY_STOP_CONDITION_PRESETS),
    "infinite": Spec(InfiniteCondition, NullParams, NULL_PRESETS),
}

VISUALIZER_REGISTRY = {
    # base realtime visualizers
    "sim": Spec(SimDvsVisualizer, SimDvsVisualizerParams, SIM_DVS_VISUALIZER_PRESETS),
    "real": Spec(RealDvsVisualizer, RealDvsVisualizerParams, REAL_DVS_VISUALIZER_PRESETS),
    "one": Spec(OneDvsVisualizer, OneDvsVisualizerParams, ONE_DVS_VISUALIZER_PRESETS),

    # workspace variants
    "sim_ws": Spec(SimDvsWorkspaceVisualizer, SimDvsWorkspaceVisualizerParams, SIM_DVS_WORKSPACE_VISUALIZER_PRESETS),
    "real_ws": Spec(RealDvsWorkspaceVisualizer, RealDvsWorkspaceVisualizerParams, REAL_DVS_WORKSPACE_VISUALIZER_PRESETS),
    
    # 3d animation
    "3d": Spec(Visualizer3D, Visualizer3DParams, VISUALIZER_3D_PRESETS),
}

PROGRESS_REGISTRY = {
    "default": Spec(ConsoleProgress, ProgressParams, PROGRESS_PRESETS),
}

PACING_REGISTRY = {
    "realtime": Spec(RealTimePacing, RealTimePacingParams, REALTIME_PACING_PRESETS),
    "null":  Spec(NoPacing, NullParams, NULL_PRESETS),
}

SCHEDULER_REGISTRY = {
    "realtime": Spec(Scheduler, SchedulerParams, SCHEDULER_PRESETS),
}

EXPERIMENT_REGISTRY = {
    "default": Spec(
        cls      = Experiment,
        Params   = ExperimentParams,
        Presets  = EXPERIMENT_PRESETS,
        registries = {
            "system":          SYSTEM_REGISTRY,
            "logger":          LOGGER_REGISTRY,
            "stop_condition":  STOP_CONDITION_REGISTRY,
            "visualizer":      VISUALIZER_REGISTRY,
            "progress":        PROGRESS_REGISTRY,
            "pacing":          PACING_REGISTRY,
            "scheduler":       SCHEDULER_REGISTRY,
        }
    )
}

# VISION_INTERFACE_REGISTRY + assemble_vision_backend merge interface + LINE_ALGO_REGISTRY + REG_MODEL_REGISTRY

Some presets

In [ ]:


VISION_PRESETS = {
    "sim_analytic": {
        "interface": "sim_analytic:default",
        "algo":      "hough:default",
        "reg_model": "none:default",
    },
    "sim_dvs_hough": {
        "interface": "sim_dvs:default",
        "algo":      "hough:default",
        "reg_model": "simple:default",
    },
    "sim_dvs_sam": {"base": "sim_dvs_hough", "algo": "sam:default"},
    "real_dvs":    {"base": "sim_dvs_hough",  "interface": "real_dvs:default"},
}